# 01 Data Quality Audit


In [ ]:
# khai báo thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Kiểm tra đường dẫn thư mục hiện tại
DATA_PATH = Path("../data/games.csv")
REPORT_PATH = Path("../reports/games_data_quality_audit.xlsx")

# Tạo thư mục reports nếu chưa tồn tại.
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Đường dẫn dữ liệu đang kiểm tra:", DATA_PATH.resolve())
print("Đường dẫn báo cáo dự kiến:", REPORT_PATH.resolve())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy data/games.csv. "
        "Hãy kiểm tra cấu trúc thư mục hoặc điều chỉnh DATA_PATH."
    )

In [ ]:
# Đọc file raw và kiểm tra số hàng cột
df_raw = pd.read_csv(DATA_PATH, encoding="utf-8", low_memory=False)

print("Đọc file thành công.")
print(f"Số dòng: {df_raw.shape[0]:,}")
print(f"Số cột: {df_raw.shape[1]:,}")
print(f"Kích thước dữ liệu: {df_raw.shape}")

display(df_raw.head())
display(df_raw.sample(min(5, len(df_raw)), random_state=42) if len(df_raw) else df_raw)

In [ ]:
# thông tin dataset raw
print("----- THÔNG TIN DATASET -----")
print("Số dòng:", len(df_raw))
print("Số cột:", len(df_raw.columns))
print("Tên cột:", df_raw.columns.tolist())
print("Dung lượng bộ nhớ (MB):",
      round(df_raw.memory_usage(deep=True).sum() / (1024 ** 2), 2))

df_raw.info()

In [ ]:
# check missing values
overview = pd.DataFrame({
    "Column": df_raw.columns,
    "Data_Type": [str(df_raw[col].dtype) for col in df_raw.columns],
    "Non_Null_Count": [int(df_raw[col].notna().sum()) for col in df_raw.columns],
    "Missing_Count": [int(df_raw[col].isna().sum()) for col in df_raw.columns],
    "Missing_Rate_%": [
        round(float(df_raw[col].isna().mean() * 100), 2)
        for col in df_raw.columns
    ],
    "Unique_Values": [
        int(df_raw[col].nunique(dropna=True)) for col in df_raw.columns
    ],
    "Unique_Rate_%": [
        round(float(df_raw[col].nunique(dropna=True) / max(len(df_raw), 1) * 100), 2)
        for col in df_raw.columns
    ]
})

display(overview)

In [ ]:
missing_report = pd.DataFrame({
    "Missing_Count": df_raw.isna().sum(),
    "Missing_Rate_%": (df_raw.isna().mean() * 100).round(2)
})

missing_report["Review_Flag"] = np.select(
    [
        missing_report["Missing_Count"].eq(0),
        missing_report["Missing_Rate_%"].le(5),
        missing_report["Missing_Rate_%"].le(30)
    ],
    [
        "No pandas-recognized missing values",
        "Review: low missing rate",
        "Review: moderate missing rate"
    ],
    default="Review: high missing rate"
)

missing_report = missing_report.sort_values(
    "Missing_Rate_%", ascending=False
)

display(missing_report)

In [ ]:
# check for duplicates
duplicate_extra_count = int(df_raw.duplicated(keep="first").sum())
duplicate_all_mask = df_raw.duplicated(keep=False)
duplicate_involved_count = int(duplicate_all_mask.sum())

duplicate_summary = pd.DataFrame([{
    "Total_Rows": len(df_raw),
    "Duplicate_Extra_Rows": duplicate_extra_count,
    "Rows_Involved_In_Duplicate_Groups": duplicate_involved_count,
    "Duplicate_Extra_Rate_%": round(
        duplicate_extra_count / max(len(df_raw), 1) * 100, 2
    )
}])

print("Tóm tắt duplicate rows:")
display(duplicate_summary)

print("Ví dụ các dòng thuộc nhóm trùng hoàn toàn:")
display(df_raw.loc[duplicate_all_mask].head(20))

In [ ]:
import warnings
warnings.filterwarnings("ignore")
 
try:
    display  # có sẵn trong Jupyter
except NameError:
    def display(x):
        print(x)
    df_typed = df_raw.copy()
type_issues = []
 
for col in df_typed.select_dtypes(include="object").columns:
    s = df_typed[col].dropna().astype(str).str.strip()
    if s.empty:
        continue
 
    # Thử ép sang số (bỏ dấu phẩy, khoảng trắng, $, €, %)
    cleaned = s.str.replace(r"[,\s$€%]", "", regex=True)
    num = pd.to_numeric(cleaned, errors="coerce")
    if num.notna().mean() > 0.8:
        type_issues.append({
            "Column": col,
            "Suggested_Type": "numeric",
            "Parse_Rate": round(num.notna().mean(), 3),
            "Non_Convertible": s[num.isna()].unique()[:5].tolist(),
        })
        df_typed[col] = pd.to_numeric(
            df_typed[col].astype(str).str.strip()
            .str.replace(r"[,\s$€%]", "", regex=True),
            errors="coerce",
        )
        continue
 
    # Thử ép sang ngày tháng
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if dt.notna().mean() > 0.8:
        type_issues.append({
            "Column": col,
            "Suggested_Type": "datetime",
            "Parse_Rate": round(dt.notna().mean(), 3),
            "Non_Convertible": s[dt.isna()].unique()[:5].tolist(),
        })
        df_typed[col] = pd.to_datetime(df_typed[col], errors="coerce", dayfirst=True)
 
print("\nCác cột bị đọc sai kiểu (đã thử ép sang df_typed):")
display(pd.DataFrame(type_issues))

In [ ]:
text_issues = []
for col in df_typed.select_dtypes(include="object").columns:
    s = df_typed[col].dropna().astype(str)
    if s.empty:
        continue
    text_issues.append({
        "Column": col,
        "Khoang_Trang_Thua": int((s != s.str.strip()).sum()),
        "Lan_Hoa_Thuong": int(s.nunique() - s.str.lower().str.strip().nunique()),
    })
 
print("\nĐịnh dạng text không nhất quán:")
display(pd.DataFrame(text_issues))

In [ ]:
# outliers detection
numeric_cols = df_raw.select_dtypes(include="number").columns.tolist()
outlier_records = []
outlier_examples = {}

for col in numeric_cols:
    series = df_raw[col].dropna()

    # Bỏ qua cột không có giá trị hợp lệ để tính toán.
    if series.empty:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (
        df_raw[col].notna()
        & ((df_raw[col] < lower_bound) | (df_raw[col] > upper_bound))
    )

    outlier_records.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": int(mask.sum()),
        "Outlier_Rate_%": round(float(mask.mean() * 100), 2),
        "IQR_Zero_Flag": bool(iqr == 0)
    })

    # Lưu một số ví dụ để điều tra, không xóa các dòng này.
    outlier_examples[col] = df_raw.loc[mask].head(10)

outlier_report = pd.DataFrame(outlier_records)
display(outlier_report)

# Xem ví dụ outlier của một cột bằng cách thay tên cột bên dưới:
# display(outlier_examples.get("column_name"))

In [ ]:
# kiểm tra các cột số


if numeric_cols:
    numeric_stats = df_raw[numeric_cols].describe(
        percentiles=[0.25, 0.50, 0.75]
    ).T

    numeric_stats["Missing_Count"] = df_raw[numeric_cols].isna().sum()
    numeric_stats["Zero_Count"] = df_raw[numeric_cols].eq(0).sum()
    numeric_stats["Negative_Count"] = df_raw[numeric_cols].lt(0).sum()
    numeric_stats["Skewness"] = df_raw[numeric_cols].skew().round(3)

    display(numeric_stats)
else:
    numeric_stats = pd.DataFrame()
    print("Dataset không có cột số được pandas nhận diện.")